<a href="https://colab.research.google.com/github/hmontUSC/movie-recommender-svd/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 2: Honest Testing and Baselines to Beat

## Part A: Load the MovieLens ratings

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

This code downloads and unzips the MovieLens 100K data only if it is not already present, then loads all 100,000 ratings into a Pandas table with one row per rating.

## Part B: Make an honest train/test split

In [ ]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    """Hide ratings without removing a user's or movie's last training rating."""
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()


train, test = split_ratings(ratings)
print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

This code randomly hides about 10% of the ratings as a test set, while guaranteeing that every user and movie keeps at least one training rating, and it returns the training and test tables.

## Part C: Build the training matrix

In [ ]:
N_USERS, N_ITEMS = 943, 1682


def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R


R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

This code converts the training table into a 943 × 1682 user-by-movie matrix with `np.nan` in every unknown position, and counts how many known ratings it contains.

In [ ]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This code pulls out the true star values of the hidden test ratings along with their matrix row (`u`) and column (`i`) positions, so predictions can be compared against them later.

## Part D: Score predictions with RMSE

In [ ]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

This function computes the root mean squared error between predicted and true ratings, so a smaller value means more accurate predictions.

## Part E: Build simple baselines

In [ ]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print("Overall average:", round(mu, 3))

This code computes the overall average rating, each user's average, and each movie's average from the training matrix, falling back to the overall average for movies with no training ratings.

In [ ]:
scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["movie average"] = rmse(item_mean[i], truth)

for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

This code predicts each hidden rating with four simple rules (always 3, overall average, user average, movie average) and prints the test RMSE of each rule.

## Part F: Combine user and movie effects

In [ ]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)

    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]

    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])

    return mu, b_user, b_item

This function estimates the overall mean rating plus a per-user effect and a per-movie effect, where each effect is the average amount that user or movie deviates from the mean.

In [ ]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores["user + movie effects"] = rmse(pred, truth)
print(pd.Series(scores, name="test RMSE").round(4).sort_values())

This code predicts each hidden rating as the overall mean plus the user and movie effects, clips the result to the 1–5 star range, and prints all five RMSE scores sorted from best to worst.

# AI Disclosure

All code in this notebook was provided by the Milestone 2.pdf document on the course website, with alterations made by me (Henry Monteith) where I felt necessary. 

All code explanations were written by me. Claude Sonnet 5.5 was used for validation of my explanations and for verification of the code itself.